# Reflex : fine-tune de Laya (Kaggle, GPU T4)

**Avant de lancer :**
1. Sur ton Mac : `python scripts/build_laya_dataset.py --per-class 3000` → crée `data/laya/` (train, val, test, eval_laya.py).
2. Kaggle → *Datasets* → *New Dataset* → glisse le contenu de `data/laya/`, nomme-le **reflex-laya-data** et laisse-le en **Private** (données personnelles réelles : ne pas le publier).
3. Ici : *Add Input* → ton dataset ; *Settings* → **Accelerator : GPU T4 x2**, **Internet : On**.
4. *Run All*. À la fin, télécharge `laya-reflex.zip` (onglet *Output*).

In [ ]:
!pip install -q laya
import glob, os
DATA = os.path.dirname(glob.glob('/kaggle/input/**/train.jsonl', recursive=True)[0])
OUT = '/kaggle/working/laya-reflex'
print('Données :', DATA, os.listdir(DATA))
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Vérifier le format (aucun entraînement)

In [ ]:
!laya-train --data {DATA}/train.jsonl --base multilingual --dry-run

## 2. Score du modèle de base, avant fine-tune (pour comparer)

In [ ]:
!python {DATA}/eval_laya.py --model multilingual --device cuda --data {DATA}/test.jsonl --out /kaggle/working/report_base.json

## 3. Fine-tune

Le multilingue (mmBERT, 322M) lit le français. 3 époques suffisent pour commencer ;
`--shuffle-options` mélange l'ordre des deux options pour que le modèle ne retienne pas leur position.

In [ ]:
!laya-train --data {DATA}/train.jsonl --eval {DATA}/val.jsonl --base multilingual --out {OUT} --epochs 3 --micro-batch 16 --grad-accum 4 --shuffle-options --device cuda --seed 42

## 4. Évaluation finale et choix des seuils de la cascade

In [ ]:
tests = ' '.join(f'--data {p}' for p in sorted(glob.glob(f'{DATA}/test*.jsonl')))
!python {DATA}/eval_laya.py --model {OUT} --device cuda {tests} --out /kaggle/working/report_finetuned.json

## 5. Récupérer le modèle

In [ ]:
!cd /kaggle/working && zip -qr laya-reflex.zip laya-reflex report_base.json report_finetuned.json && ls -lh laya-reflex.zip